# LeapQuant Flush — Pure-PyTorch Port (Colab Verification)

Self-contained validation of **P1 (pure-torch reference port) + P2 (repository judge)** for the LeapQuant window-boundary flush kernel.

- **Reference**: verbatim from `gdn|kda/definition.json` (fp32 math truth).
- **Port**: `gdn_torch.py` / `kda_torch.py` (literal translation, no cutlass/tilelang/B300).
- **Judge**: mirrors `{gdn,kda}/benchmark.py` — decode the new checkpoint back to a V×K state and compare, head by head, against the fp64-rebuilt state `S`. PASS ⇔ finite, codes in [-127,127], and `err_cand ≤ (1+1%)·err_ref` (or `≤1e-3·|S|` when the reference error is below `1e-4·|S|`).

> Only dependency is `torch`, which Colab ships with (GPU runtime recommended). Zero file/network access needed.

In [ ]:
import torch
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device", DEV, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
BATCHES = [16, 32, 64, 128]   # add 256, 512 to match the repository workloads (slower on CPU)

## 1. Load the mathematical reference (verbatim from definition.json)

In [ ]:
GDN_REF = '''# SPDX-License-Identifier: Apache-2.0
import torch

QMAX = 127.0
EPS = 1e-12


def _mgs(m):
    """Modified Gram-Schmidt on the columns of m, projections applied twice ("twice is enough"): a column that is
    numerically dependent on the previous ones (a rank-deficient head) still comes out orthogonal to them."""
    cols = []
    for j in range(m.shape[-1]):
        v = m[..., j]
        for _ in range(2):
            for c in cols:
                v = v - c * (c * v).sum(-1, keepdim=True)
        cols.append(v / v.norm(dim=-1, keepdim=True).clamp_min(1e-30))
    return torch.stack(cols, -1)


def run(codes, s_k, s_v, u, q, kbuf, ubuf, w, p, q0):
    """LeapQuant window flush: fold the buffered updates of one window into a new quantized checkpoint.

    codes int8 [B, H, V, K], s_k f32 [B, H, K], s_v f32 [B, H, V]: smoothed int8 residual of the old checkpoint
    u f16 [B, H, R, K], q f16 [B, H, R, V]: the R Compensator Tokens of the old checkpoint (rank-R part = q^T u)
    kbuf bf16 [B, H, L, K], ubuf bf16 [B, H, L, V], w f32 [B, H, L], p f32 [B, H]: the window's buffered updates
    q0 f32 [V, R]: fixed start of the subspace iteration
    Returns the new (codes, s_k, s_v, u, q).  fp32 arithmetic; the order of operations is not part of the contract.
    """
    # 1. rebuild the fp32 state: S = p * (q^T u + dequant(codes)) + sum_j w_j ubuf_j kbuf_j^T      [V, K]
    s_old = torch.einsum("bhrv,bhrk->bhvk", q.float(), u.float())
    s_old = s_old + codes.float() * (s_k[..., None, :] / QMAX) * s_v[..., :, None]
    s = p[..., None, None] * s_old + torch.einsum("bhj,bhjv,bhjk->bhvk", w, ubuf.float(), kbuf.float())
    # 2. Compensator Tokens: ONE round of subspace iteration from q0, stored as fp16
    x = s.transpose(-1, -2)                      # [K, V]
    qk = _mgs(x @ q0)                            # [K, R]
    qv = _mgs(x.transpose(-1, -2) @ qk)          # [V, R], orthonormal columns
    uk = x @ qv                                  # [K, R]
    u_new = uk.transpose(-1, -2).to(torch.float16)
    q_new = qv.transpose(-1, -2).to(torch.float16)
    # 3. residual, smoothing scales (column scale = sqrt of the mean |E| over rows, row scale = max of the smoothed row), int8
    e = s - torch.einsum("bhrv,bhrk->bhvk", q_new.float(), u_new.float())
    sk = e.abs().mean(-2).sqrt().clamp_min(EPS)
    sv = (e.abs() / sk[..., None, :]).amax(-1).clamp_min(EPS)
    c = torch.round(e / (sk[..., None, :] * sv[..., :, None]) * QMAX).clamp(-QMAX, QMAX).to(torch.int8)
    return c, sk, sv, u_new, q_new
'''
KDA_REF = '''# SPDX-License-Identifier: Apache-2.0
import torch

QMAX = 127.0
EPS = 1e-12


def _mgs(m):
    """Modified Gram-Schmidt on the columns of m, projections applied twice ("twice is enough"): a column that is
    numerically dependent on the previous ones (a rank-deficient head) still comes out orthogonal to them."""
    cols = []
    for j in range(m.shape[-1]):
        v = m[..., j]
        for _ in range(2):
            for c in cols:
                v = v - c * (c * v).sum(-1, keepdim=True)
        cols.append(v / v.norm(dim=-1, keepdim=True).clamp_min(1e-30))
    return torch.stack(cols, -1)


def run(codes, s_k, s_v, u, q, kbuf, ubuf, gbuf, w, pcum, q0):
    """LeapQuant window flush for Kimi Delta Attention (per-key-channel gate): fold the buffered updates of one window into a new
    quantized checkpoint.

    codes int8 [B, H, V, K], s_k f32 [B, H, K], s_v f32 [B, H, V]: smoothed int8 residual of the old checkpoint
    u f16 [B, H, R, K], q f16 [B, H, R, V]: the R Compensator Tokens of the old checkpoint (rank-R part = q^T u)
    kbuf bf16 [B, H, L, K], ubuf bf16 [B, H, L, V]: the window's buffered updates; w f32 [B, H, L]: their weights
    gbuf f16 [B, H, L, K]: per-channel decay factor of the step that appended each entry; pcum f32 [B, H, K]: log-gate summed
        over the window (the checkpoint's decay is exp(pcum))
    q0 f32 [V, R]: fixed start of the subspace iteration
    Returns the new (codes, s_k, s_v, u, q).  fp32 arithmetic; the order of operations is not part of the contract.
    """
    # 1. rebuild the fp32 state: S = (q^T u + dequant(codes)) diag(exp(pcum)) + sum_j w_j ubuf_j (kbuf_j * D_j)^T     [V, K]
    #    with D_j = prod_{i > j} gbuf_i (the decay entry j has seen since it was appended)
    s_old = torch.einsum("bhrv,bhrk->bhvk", q.float(), u.float())
    s_old = s_old + codes.float() * (s_k[..., None, :] / QMAX) * s_v[..., :, None]
    after = torch.flip(torch.cumprod(torch.flip(gbuf.float(), [-2]), -2), [-2])                 # prod_{i >= j}
    D = torch.cat([after[..., 1:, :], torch.ones_like(after[..., :1, :])], -2)                  # prod_{i > j}
    s = s_old * torch.exp(pcum)[..., None, :] + torch.einsum("bhj,bhjv,bhjk->bhvk", w, ubuf.float(), kbuf.float() * D)
    # 2. Compensator Tokens: ONE round of subspace iteration from q0, stored as fp16
    x = s.transpose(-1, -2)                      # [K, V]
    qk = _mgs(x @ q0)                            # [K, R]
    qv = _mgs(x.transpose(-1, -2) @ qk)          # [V, R], orthonormal columns
    uk = x @ qv                                  # [K, R]
    u_new = uk.transpose(-1, -2).to(torch.float16)
    q_new = qv.transpose(-1, -2).to(torch.float16)
    # 3. residual, smoothing scales (column scale = sqrt of the mean |E| over rows, row scale = max of the smoothed row), int8
    e = s - torch.einsum("bhrv,bhrk->bhvk", q_new.float(), u_new.float())
    sk = e.abs().mean(-2).sqrt().clamp_min(EPS)
    sv = (e.abs() / sk[..., None, :]).amax(-1).clamp_min(EPS)
    c = torch.round(e / (sk[..., None, :] * sv[..., :, None]) * QMAX).clamp(-QMAX, QMAX).to(torch.int8)
    return c, sk, sv, u_new, q_new
'''
_g = {}; exec(GDN_REF, _g); ref_gdn = _g["run"]
_k = {}; exec(KDA_REF, _k); ref_kda = _k["run"]
print("reference loaded: GDN, KDA")

## 2. Pure-PyTorch port (P1)

In [ ]:
_gport = {}; exec('''# SPDX-License-Identifier: Apache-2.0
"""Pure-PyTorch port of the LeapQuant GDN window flush (arXiv:2609.38166).

Faithful literal translation of the mathematical reference in gdn/definition.json
(codes int8 smoothed residual + R Compensator Tokens -> one window boundary re-quantization).
No cutlass / tilelang / B300 dependency; runs on any torch device (CUDA or CPU).
Interface mirrors the definition contract:
    run(codes, s_k, s_v, u, q, kbuf, ubuf, w, p, q0) -> (codes_new, s_k_new, s_v_new, u_new, q_new)
Arithmetic is fp32; operation order is deliberately identical to the reference so that
the P2 judge (decode checkpoint vs fp64-rebuilt state) sees the maximum slack.
"""
import torch

QMAX = 127.0
EPS = 1e-12


def _mgs(m):
    """Modified Gram-Schmidt on the columns of m, projections applied twice ("twice is enough"):
    a numerically dependent column (rank-deficient head) still comes out orthogonal to the rest."""
    cols = []
    for j in range(m.shape[-1]):
        v = m[..., j]
        for _ in range(2):
            for c in cols:
                v = v - c * (c * v).sum(-1, keepdim=True)
        cols.append(v / v.norm(dim=-1, keepdim=True).clamp_min(1e-30))
    return torch.stack(cols, -1)


def rebuild_state(codes, s_k, s_v, u, q, kbuf, ubuf, w, p):
    """1. Rebuild the fp32 state S = p*(q^T u + dequant(codes)) + sum_j w_j ubuf_j kbuf_j^T  [V, K]."""
    s_old = torch.einsum("bhrv,bhrk->bhvk", q.float(), u.float())
    s_old = s_old + codes.float() * (s_k[..., None, :] / QMAX) * s_v[..., :, None]
    s = p[..., None, None] * s_old + torch.einsum("bhj,bhjv,bhjk->bhvk", w, ubuf.float(), kbuf.float())
    return s


def subspace_iteration(s, q0):
    """2. ONE round of subspace iteration from q0 -> new R Compensator Tokens, stored as fp16."""
    x = s.transpose(-1, -2)                       # [K, V]
    qk = _mgs(x @ q0)                             # [K, R]
    qv = _mgs(x.transpose(-1, -2) @ qk)           # [V, R], orthonormal columns
    uk = x @ qv                                   # [K, R]
    u_new = uk.transpose(-1, -2).to(torch.float16)
    q_new = qv.transpose(-1, -2).to(torch.float16)
    return u_new, q_new


def quantize(s, u_new, q_new):
    """3. Residual E = S - q_new^T u_new, smoothing scales, int8 codes in [-127, 127]."""
    e = s - torch.einsum("bhrv,bhrk->bhvk", q_new.float(), u_new.float())
    sk = e.abs().mean(-2).sqrt().clamp_min(EPS)            # column scale: sqrt of the mean |E| over rows
    sv = (e.abs() / sk[..., None, :]).amax(-1).clamp_min(EPS)  # row scale: max of the smoothed row
    c = torch.round(e / (sk[..., None, :] * sv[..., :, None]) * QMAX).clamp(-QMAX, QMAX).to(torch.int8)
    return c, sk, sv


def run(codes, s_k, s_v, u, q, kbuf, ubuf, w, p, q0):
    """LeapQuant window flush for the scalar-gate (GDN) linear attention.

    codes int8 [B, H, V, K]; s_k/s_v f32 [B, H, K]/[B, H, V] (smoothed int8 residual scales)
    u/q f16 [B, H, R, K]/[B, H, R, V]  (old Compensator Tokens; rank-R part = q^T u)
    kbuf/ubuf bf16 [B, H, L, K]/[B, H, L, V]; w f32 [B, H, L]; p f32 [B, H]  (window's buffered updates)
    q0 f32 [V, R]  (fixed start of the subspace iteration)
    """
    s = rebuild_state(codes, s_k, s_v, u, q, kbuf, ubuf, w, p)
    u_new, q_new = subspace_iteration(s, q0)
    c, sk, sv = quantize(s, u_new, q_new)
    return c, sk, sv, u_new, q_new
''', _gport)
gdn_torch_run = _gport["run"]
print("gdn port loaded")

In [ ]:
_kport = {}; exec('''# SPDX-License-Identifier: Apache-2.0
"""Pure-PyTorch port of the LeapQuant KDA window flush (arXiv:2609.38166, per-key-channel gate).

Faithful literal translation of the mathematical reference in kda/definition.json.
No cutlass / tilelang / B300 dependency; runs on any torch device (CUDA or CPU).
Interface mirrors the definition contract:
    run(codes, s_k, s_v, u, q, kbuf, ubuf, gbuf, w, pcum, q0) -> (codes_new, s_k_new, s_v_new, u_new, q_new)
Differs from the GDN port only in the state rebuild: the checkpoint is decayed per key channel
by exp(pcum) and each buffered rank-1 update by the product of decay factors seen since append.
"""
import torch

QMAX = 127.0
EPS = 1e-12


def _mgs(m):
    """Modified Gram-Schmidt on the columns of m, projections applied twice (rank-deficient heads stay orthogonal)."""
    cols = []
    for j in range(m.shape[-1]):
        v = m[..., j]
        for _ in range(2):
            for c in cols:
                v = v - c * (c * v).sum(-1, keepdim=True)
        cols.append(v / v.norm(dim=-1, keepdim=True).clamp_min(1e-30))
    return torch.stack(cols, -1)


def decay_matrix(gbuf):
    """D_j = prod_{i > j} gbuf_i  (decay entry j has seen since it was appended)."""
    after = torch.flip(torch.cumprod(torch.flip(gbuf.float(), [-2]), -2), [-2])   # prod_{i >= j}
    return torch.cat([after[..., 1:, :], torch.ones_like(after[..., :1, :])], -2)  # prod_{i > j}


def rebuild_state(codes, s_k, s_v, u, q, kbuf, ubuf, gbuf, w, pcum):
    """1. Rebuild fp32 state S = (q^T u + dequant(codes)) diag(exp(pcum)) + sum_j w_j ubuf_j (kbuf_j * D_j)^T."""
    s_old = torch.einsum("bhrv,bhrk->bhvk", q.float(), u.float())
    s_old = s_old + codes.float() * (s_k[..., None, :] / QMAX) * s_v[..., :, None]
    D = decay_matrix(gbuf)
    s = s_old * torch.exp(pcum)[..., None, :] + torch.einsum("bhj,bhjv,bhjk->bhvk", w, ubuf.float(), kbuf.float() * D)
    return s


def subspace_iteration(s, q0):
    """2. ONE round of subspace iteration from q0 -> new R Compensator Tokens, stored as fp16."""
    x = s.transpose(-1, -2)                       # [K, V]
    qk = _mgs(x @ q0)                             # [K, R]
    qv = _mgs(x.transpose(-1, -2) @ qk)           # [V, R], orthonormal columns
    uk = x @ qv                                   # [K, R]
    u_new = uk.transpose(-1, -2).to(torch.float16)
    q_new = qv.transpose(-1, -2).to(torch.float16)
    return u_new, q_new


def quantize(s, u_new, q_new):
    """3. Residual E = S - q_new^T u_new, smoothing scales, int8 codes in [-127, 127]."""
    e = s - torch.einsum("bhrv,bhrk->bhvk", q_new.float(), u_new.float())
    sk = e.abs().mean(-2).sqrt().clamp_min(EPS)
    sv = (e.abs() / sk[..., None, :]).amax(-1).clamp_min(EPS)
    c = torch.round(e / (sk[..., None, :] * sv[..., :, None]) * QMAX).clamp(-QMAX, QMAX).to(torch.int8)
    return c, sk, sv


def run(codes, s_k, s_v, u, q, kbuf, ubuf, gbuf, w, pcum, q0):
    """LeapQuant window flush for Kimi Delta Attention (per-key-channel gate).

    codes int8 [B, H, V, K]; s_k/s_v f32 [B, H, K]/[B, H, V]
    u/q f16 [B, H, R, K]/[B, H, R, V] (old Compensator Tokens)
    kbuf/ubuf bf16 [B, H, L, K]/[B, H, L, V]; gbuf f16 [B, H, L, K] (per-channel decay factor)
    w f32 [B, H, L]; pcum f32 [B, H, K] (log-gate summed over the window; checkpoint decay = exp(pcum))
    q0 f32 [V, R] (fixed start of the subspace iteration)
    """
    s = rebuild_state(codes, s_k, s_v, u, q, kbuf, ubuf, gbuf, w, pcum)
    u_new, q_new = subspace_iteration(s, q0)
    c, sk, sv = quantize(s, u_new, q_new)
    return c, sk, sv, u_new, q_new
''', _kport)
kda_torch_run = _kport["run"]
print("kda port loaded")

## 3. Judge + input construction (P2, mirrors benchmark.py)

In [ ]:

# --------------------------------------------------------------------------- judge / inputs (mirrors {gdn,kda}/benchmark.py, device-parameterized)
QMAX, EPS = 127.0, 1e-12
REL_TOL, FLOOR, CAP = 0.01, 1e-4, 1e-3
HV, K, V, L, R = 32, 128, 128, 16, 4

def decode(codes, s_k, s_v, u, q):
    return (torch.einsum("bhrv,bhrk->bhvk", q.double(), u.double())
            + codes.double() * (s_k.double()[..., None, :] / QMAX) * s_v.double()[..., :, None])

def judge(out, ref, S):
    finite = all(bool(torch.isfinite(t.float()).all()) for t in out[1:]); inrange = bool((out[0].int().abs() <= 127).all())
    e_out = (decode(*out) - S).abs().mean((0, 2, 3)); e_ref = (decode(*ref) - S).abs().mean((0, 2, 3)); mag = S.abs().mean((0, 2, 3))
    tight = (e_ref >= FLOOR * mag) & (e_ref > 0)
    ok = finite and inrange and bool((e_out[tight] <= (1 + REL_TOL) * e_ref[tight]).all()) and bool((e_out[~tight] <= CAP * mag[~tight]).all())
    ratio = (e_out[tight] / e_ref[tight]).max().item() if tight.any() else 0.0
    return ok, ratio

def gdn_update_buffer(bs, dev, g):
    rn = lambda *s: torch.randn(*s, device=dev, generator=g)
    kbuf = (rn(bs, HV, L, K) * 0.3).to(torch.bfloat16); ubuf = (rn(bs, HV, L, V) * 0.3).to(torch.bfloat16)
    w = torch.rand(bs, HV, L, device=dev, generator=g); p = torch.rand(bs, HV, device=dev, generator=g) * 0.5 + 0.5
    return kbuf, ubuf, w, p

def gdn_structured(bs, dev, seed=0):
    g = torch.Generator(device=dev).manual_seed(seed)
    rn = lambda *s: torch.randn(*s, device=dev, generator=g)
    codes = (rn(bs, HV, V, K) * 40).round().clamp(-127, 127).to(torch.int8)
    s_k = torch.rand(bs, HV, K, device=dev, generator=g) * 0.2 + 0.05; s_v = torch.rand(bs, HV, V, device=dev, generator=g) * 0.5 + 0.1
    s_v[..., 5] *= 30
    u = (rn(bs, HV, R, K) * 0.5).to(torch.float16); q = (rn(bs, HV, R, V) * 0.5).to(torch.float16)
    kbuf, ubuf, w, p = gdn_update_buffer(bs, dev, g)
    for h in (1, 2, 3, 6): codes[:, h] = 0; u[:, h] = 0; q[:, h] = 0
    w[:, 1, 1:] = 0; w[:, 2] = 0; w[:, 3, 2:] = 0; w[:, 6, 1:] = 0
    codes[:, 6] = (rn(bs, V, K) * 40).round().clamp(-127, 127).to(torch.int8); s_k[:, 6] *= 1e-6
    s_k[:, 4] *= 1e-4; u[:, 4] *= 1e-2; ubuf[:, 4] *= 1e-4
    s_k[:, 5] *= 3e2; u[:, 5] *= 17; ubuf[:, 5] *= 3e2
    q0 = torch.randn(V, R, generator=torch.Generator().manual_seed(0)).to(dev)
    return codes, s_k, s_v, u, q, kbuf, ubuf, w, p, q0

def gdn_random(bs, dev, seed=1):
    g = torch.Generator(device=dev).manual_seed(seed)
    rn = lambda *s, dt=torch.float32: torch.randn(*s, device=dev, generator=g).to(dt)
    lu = lambda *s: torch.exp(rn(*s) * 0.7)
    return (torch.randint(-127, 128, (bs, HV, V, K), device=dev, generator=g, dtype=torch.int8), lu(bs, HV, K) * 0.05, lu(bs, HV, V) * 0.3,
            rn(bs, HV, R, K, dt=torch.float16), rn(bs, HV, R, V, dt=torch.float16), rn(bs, HV, L, K, dt=torch.bfloat16),
            rn(bs, HV, L, V, dt=torch.bfloat16), torch.rand(bs, HV, L, device=dev, generator=g), torch.rand(bs, HV, device=dev, generator=g) * 0.999 + 0.001, rn(V, R))

def gdn_exact_state(inp):
    codes, s_k, s_v, u, q, kbuf, ubuf, w, p, q0 = inp
    return p.double()[..., None, None] * decode(codes, s_k, s_v, u, q) + torch.einsum("bhj,bhjv,bhjk->bhvk", w.double(), ubuf.double(), kbuf.double())

def kda_update_buffer(bs, dev, g):
    rn = lambda *s: torch.randn(*s, device=dev, generator=g)
    kbuf = (rn(bs, HV, L, K) * 0.3).to(torch.bfloat16); ubuf = (rn(bs, HV, L, V) * 0.3).to(torch.bfloat16)
    w = torch.ones(bs, HV, L, device=dev)
    gbuf = (torch.rand(bs, HV, L, K, device=dev, generator=g) * 0.4 + 0.6).half()
    pcum = gbuf.float().log().sum(-2) - torch.rand(bs, HV, K, device=dev, generator=g) * 0.5
    return kbuf, ubuf, gbuf, w, pcum

def kda_structured(bs, dev, seed=0):
    g = torch.Generator(device=dev).manual_seed(seed)
    rn = lambda *s: torch.randn(*s, device=dev, generator=g)
    codes = (rn(bs, HV, V, K) * 40).round().clamp(-127, 127).to(torch.int8)
    s_k = torch.rand(bs, HV, K, device=dev, generator=g) * 0.2 + 0.05; s_v = torch.rand(bs, HV, V, device=dev, generator=g) * 0.5 + 0.1
    s_v[..., 5] *= 30
    u = (rn(bs, HV, R, K) * 0.5).to(torch.float16); q = (rn(bs, HV, R, V) * 0.5).to(torch.float16)
    kbuf, ubuf, gbuf, w, pcum = kda_update_buffer(bs, dev, g)
    for h in (1, 2, 3, 6): codes[:, h] = 0; u[:, h] = 0; q[:, h] = 0
    w[:, 1, 1:] = 0; w[:, 2] = 0; w[:, 3, 2:] = 0; w[:, 6, 1:] = 0
    codes[:, 6] = (rn(bs, V, K) * 40).round().clamp(-127, 127).to(torch.int8); s_k[:, 6] *= 1e-6
    s_k[:, 4] *= 1e-4; u[:, 4] *= 1e-2; ubuf[:, 4] *= 1e-4
    s_k[:, 5] *= 3e2; u[:, 5] *= 17; ubuf[:, 5] *= 3e2
    pcum[:, 7] -= 200.0
    gbuf[:, 8, :, :32] = 1.0; pcum[:, 8, :32] = 0.0
    q0 = torch.randn(V, R, generator=torch.Generator().manual_seed(0)).to(dev)
    return codes, s_k, s_v, u, q, kbuf, ubuf, gbuf, w, pcum, q0

def kda_random(bs, dev, seed=1):
    g = torch.Generator(device=dev).manual_seed(seed)
    rn = lambda *s, dt=torch.float32: torch.randn(*s, device=dev, generator=g).to(dt)
    lu = lambda *s: torch.exp(rn(*s) * 0.7)
    return (torch.randint(-127, 128, (bs, HV, V, K), device=dev, generator=g, dtype=torch.int8), lu(bs, HV, K) * 0.05, lu(bs, HV, V) * 0.3,
            rn(bs, HV, R, K, dt=torch.float16), rn(bs, HV, R, V, dt=torch.float16), rn(bs, HV, L, K, dt=torch.bfloat16),
            rn(bs, HV, L, V, dt=torch.bfloat16), (torch.rand(bs, HV, L, K, device=dev, generator=g) * 0.999 + 0.001).half(),
            torch.rand(bs, HV, L, device=dev, generator=g), -torch.rand(bs, HV, K, device=dev, generator=g) * 20, rn(V, R))

def kda_exact_state(inp):
    codes, s_k, s_v, u, q, kbuf, ubuf, gbuf, w, pcum, q0 = inp
    after = torch.flip(torch.cumprod(torch.flip(gbuf.double(), [-2]), -2), [-2])
    D = torch.cat([after[..., 1:, :], torch.ones_like(after[..., :1, :])], -2)
    return decode(codes, s_k, s_v, u, q) * torch.exp(pcum.double())[..., None, :] + torch.einsum("bhj,bhjv,bhjk->bhvk", w.double(), ubuf.double(), kbuf.double() * D)

def run_check(which, port_run, ref_run, sizes, dev):
    structured, random, update, exact = (globals()[f"{which}_structured"], globals()[f"{which}_random"],
                                         globals()[f"{which}_update_buffer"], globals()[f"{which}_exact_state"])
    q0_idx = {"gdn": 9, "kda": 10}[which]
    rows = []
    for bs in sizes:
        first = structured(bs, dev, seed=bs)
        ref1 = ref_run(*[t.clone() for t in first])
        second = (*ref1, *update(bs, dev, torch.Generator(device=dev).manual_seed(bs + 1)), first[q0_idx])
        for name, inp, ref in (("structured", first, ref1), ("second_window", second, None), ("random", random(bs, dev, seed=bs + 2), None)):
            ref = ref_run(*[t.clone() for t in inp]) if ref is None else ref
            out = port_run(*[t.clone() for t in inp])
            ok, ratio = judge(out, ref, exact(inp))
            rows.append((bs, name, ok, ratio))
    return rows


## 4. Run validation (GDN, then KDA)

In [ ]:
print("=" * 62, "\nGDN pure-torch port vs reference\n", "=" * 62)
for bs, name, ok, ratio in run_check("gdn", gdn_torch_run, ref_gdn, BATCHES, DEV):
    print(f"| {bs:>4} | {name:<13} | {'PASS' if ok else 'FAIL':<5} | worst-head err ratio {ratio:.6f}")
print("\n" + "=" * 62, "\nKDA pure-torch port vs reference\n", "=" * 62)
for bs, name, ok, ratio in run_check("kda", kda_torch_run, ref_kda, BATCHES, DEV):
    print(f"| {bs:>4} | {name:<13} | {'PASS' if ok else 'FAIL':<5} | worst-head err ratio {ratio:.6f}")

## 5. Interpreting the result

- `worst-head err ratio = 1.000000` means the port reproduces the fp32 reference *exactly* (it is a faithful direct translation), so it passes the judge with maximum slack.
- `PASS` across every batch × input family = the pure-torch port is contract- and numerically-equivalent to the repository's reference on all cases (`3·len(BATCHES)` here, or all 18 when BATCHES = the 6 workloads).
- This validates P1/P2 on any torch device; for production performance (P3+) the port would need `torch.compile` / custom kernels, per the feasibility note.